# Drone Body Evolution + PPO (Colab)

Runs `7_drone_evo_rl_quintic.py` on a Colab GPU.  
Results are saved to **Google Drive** so they survive session restarts.

**Recommended runtime:** GPU → A100 (Colab Pro+) or T4 (free/Pro)

In [ ]:
# ── 1. Check what GPU we got ──────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ── 2. Mount Google Drive (results will be saved here) ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ariel_drone_evo'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

In [ ]:
# ── 3. Clone repo (drones branch) ────────────────────────────────────────────
import os
%cd /content

# Re-runnable: pull if already cloned, clone fresh otherwise.
if os.path.isdir('/content/ariel/.git'):
    print('Repo already cloned — pulling latest changes.')
    !git -C /content/ariel pull --ff-only
else:
    !git clone -b drones https://github.com/ci-group/ariel.git

%cd /content/ariel

# Symlink __data__ → Drive so all run outputs persist automatically.
!ln -sfn {DRIVE_DIR} /content/ariel/__data__
print('__data__ →', os.path.realpath('/content/ariel/__data__'))

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
# Install ariel without its heavy optional deps (mujoco-mjx/warp, nicegui,
# cadquery, etc.) then add exactly what this experiment needs.
!pip install -q -e . --no-deps
!pip install -q \
    mujoco \
    'stable-baselines3>=2.0' \
    gymnasium \
    rich \
    sqlalchemy \
    sqlmodel \
    numpy-quaternion \
    networkx \
    python-fcl
print('Done.')

In [ ]:
# ── 5. Sanity check ───────────────────────────────────────────────────────────
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

import ariel
import mujoco
import stable_baselines3 as sb3
print('ariel OK | mujoco', mujoco.__version__, '| sb3', sb3.__version__)

In [ ]:
# ── 6. Configure experiment ───────────────────────────────────────────────────
# Tune these to your GPU / time budget.

GENS      = 100        # generations
POP       = 10         # population size
PPO_STEPS = 2_000_000  # PPO timesteps per individual (more = better policy)
NUM_ENVS  = 4000       # parallel envs; increase on A100 (try 8000)
SEED      = 42
DEVICE    = 'cuda:0'   # or 'cpu' if something goes wrong

print(f'gens={GENS}  pop={POP}  ppo_steps={PPO_STEPS:,}  '
      f'num_envs={NUM_ENVS}  seed={SEED}  device={DEVICE}')

In [ ]:
# ── 7. Run ────────────────────────────────────────────────────────────────────
# Output streams directly to this cell; the DB is committed after every
# individual so a disconnected session loses at most one evaluation.
!cd /content/ariel && python examples/e_drones_ec/7_drone_evo_rl_quintic.py \
    --gens      {GENS}      \
    --pop       {POP}       \
    --ppo-steps {PPO_STEPS} \
    --num-envs  {NUM_ENVS}  \
    --seed      {SEED}      \
    --device    {DEVICE}

In [ ]:
# ── 8. Quick fitness plot (run after experiment finishes) ─────────────────────
import sqlite3, glob, os
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

# Find the latest DB in the Drive output directory.
dbs = sorted(glob.glob(f'{DRIVE_DIR}/drone_evo_rl_quintic/**/database_*.db', recursive=True))
if not dbs:
    print('No DB found yet.')
else:
    db = dbs[-1]
    print('Reading:', db)
    conn = sqlite3.connect(db)
    rows = conn.execute(
        'SELECT time_of_birth, fitness_ FROM individual '
        'WHERE fitness_ IS NOT NULL ORDER BY time_of_birth'
    ).fetchall()
    conn.close()

    gen_fits = defaultdict(list)
    for gen, fit in rows:
        gen_fits[int(gen)].append(float(fit))

    gens     = sorted(gen_fits)
    best_fit = [max(gen_fits[g]) for g in gens]
    mean_fit = [float(np.mean(gen_fits[g])) for g in gens]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(gens, best_fit, 'o-', color='steelblue', label='best')
    ax.plot(gens, mean_fit, 's--', color='tomato',   label='mean')
    ax.axhline(0, color='k', lw=0.5, ls=':')
    ax.set_xlabel('Generation'); ax.set_ylabel('Fitness (mean episode reward)')
    ax.set_title('Drone morphology evolution — fitness history')
    ax.legend(); ax.grid(True, alpha=0.4)
    plt.tight_layout(); plt.show()
    print(f'Best fitness so far: {max(best_fit):.3f}  (gen {gens[best_fit.index(max(best_fit))]})')